# Scraping Data Ulasan Play Store (Indonesia)

Notebook ini mengambil data content aplikasi Duolingo secara mandiri dari internet menggunakan `google-play-scraper`.

Target:
- Minimal 10.000 sampel
- Data disimpan ke CSV untuk dipakai pada notebook analisis sentimen

Sumber data:
- Google Play Store

In [7]:
%pip install -q google-play-scraper pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [8]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Scrapping review aplikasi Duolingo
APP_ID = "com.levelinfinite.sgameGlobal"
APP_NAME = "Honor of King"
LANG = "id"
COUNTRY = "id"
TARGET_COUNT = 10000
MAX_BATCH = 2000

OUTPUT_DIR = Path("content")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "reviews_hok_newest.csv"

print(f"Target total sampel: ~{TARGET_COUNT}".replace(",", "."))

Target total sampel: ~10000


In [9]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_count,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "app_id": app_id,
                    "app_name": app_name,
                    "review_id": r.get("reviewId"),
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                    "thumbs_up": r.get("thumbsUpCount"),
                    "at": r.get("at"),
                    "app_version": r.get("appVersion"),
                }
            )

        if token is None:
            break

    return all_rows

In [10]:
all_reviews = scrape_reviews_for_app(APP_ID, APP_NAME, target_count=TARGET_COUNT)

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal content terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()


Total content terkumpul (raw): 10.000


,app_id,app_name,review_id,user_name,score,content,thumbs_up,at,app_version
0,com.levelinfinite.sgameGlobal,Honor of King,4cd20442-e195-4c8a-92c2-114d62b9c473,Irul Aman,2,bintang 2 dulu soalnya belum main wkwkwk🗿,0,2026-04-29 11:17:48,NaN
1,com.levelinfinite.sgameGlobal,Honor of King,35bd2c67-c959-4065-b487-32be07dbd96d,Best Royal,1,NIRU NIRU ML HOOOOOO SKIN GOJO NYA PASTI NIRU ML SOALNYA ULTINYA SAMA UUUUUHHHHH NIRU2,0,2026-04-29 10:37:25,NaN
2,com.levelinfinite.sgameGlobal,Honor of King,7c170566-8968-4b89-b2a1-1e27e0080628,Lord Kaneki,5,Game yang sangat bagus 👌,0,2026-04-29 10:27:49,11.3.1.9
3,com.levelinfinite.sgameGlobal,Honor of King,de2aa256-bd78-418d-a2df-ee15340a9d20,Rudi Lis,2,"mulai hari detik ini aku unistal hok dan ga akan main lagi ni game ,sakit kepala ku kambuh karena game ini matcmakin...",0,2026-04-29 10:13:21,NaN
4,com.levelinfinite.sgameGlobal,Honor of King,11008d28-170f-4b33-ad6f-71acb4b8b491,EDIT JSON PROGRAMER,2,"halo kembali, aku mengalami bug (aku sedang bermain lubu sebelum nya aku memakai skin lubu tapi pas masuk ke dalam p...",190,2026-04-29 09:52:47,11.3.1.9


In [11]:
df = raw_df.copy()

print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "review_id", "app_id", "app_name", "user_name", "score", "content", "thumbs_up", "at", "app_version"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV.resolve()}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Distribusi skor:


score
1    2809
2     514
3     584
4     800
5    5293
Name: count, dtype: int64


File CSV tersimpan di: C:\Users\DianErdiana\dianerdiana-learning\machine-learning-dicoding\submission-deep-learning-01\content\reviews_hok_newest.csv
Jumlah sampel akhir: 10.000


In [12]:
df.sample(5, random_state=42)[["app_name", "score", "content"]]

,app_name,score,content
6252,Honor of King,5,"bagus banget,ekhem min,tambahin dong kata kata nyaa"
4684,Honor of King,4,"lumayan suka gitu aj,sering dapat Hero gratis gitu, perfect lah"
1731,Honor of King,1,"baru saja dipuji.. setelah update,penyakit timbul lagi kaya moba sebelah ""MACHTMAKING AMBURADUL"" bukannya ngilangin ..."
4742,Honor of King,1,"disappointed with the match making, enemy team has skilled Players while all i get are feeders."
4521,Honor of King,5,tolong pembuat game ini embnya minta direndahkan soalnya banyak bocil yg mau main game ini karena embnya terlalu ged...
